In [ ]:
import sys

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

In [ ]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [ ]:
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

PROJECT_PATH = Path.cwd()
if PROJECT_PATH.name == "ingestion":
    PROJECT_PATH = PROJECT_PATH.parent

CSV_PATH = (
    PROJECT_PATH
    / "data"
    / "raw"
    / "Credit_Risk_Dataset.csv"
)

print("Project path:")
print(PROJECT_PATH)

print("\nCSV path:")
print(CSV_PATH)

print("\nFile exists:")
print(CSV_PATH.exists())


In [ ]:
df = pd.read_csv(
    CSV_PATH,
    dtype=str,
    keep_default_na=False
)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

In [ ]:
EXPECTED_COLUMNS = [
    "client_ID",
    "person_age",
    "person_income",
    "person_home_ownership",
    "person_emp_length",
    "loan_intent",
    "loan_grade",
    "loan_amnt",
    "loan_int_rate",
    "loan_status",
    "loan_percent_income",
    "cb_person_default_on_file",
    "cb_person_cred_hist_length",
    "gender",
    "marital_status",
    "education_level",
    "country",
    "state",
    "city",
    "city_latitude",
    "city_longitude",
    "employment_type",
    "loan_term_months",
    "loan_to_income_ratio",
    "other_debt",
    "debt_to_income_ratio",
    "open_accounts",
    "credit_utilization_ratio",
    "past_delinquencies",
]

if df.columns.tolist() == EXPECTED_COLUMNS:
    print("Column validation: PASSED")
else:
    print("Column validation: FAILED")

    print("\nMissing columns:")
    print(set(EXPECTED_COLUMNS) - set(df.columns))

    print("\nUnexpected columns:")
    print(set(df.columns) - set(EXPECTED_COLUMNS))

In [ ]:
df.head()

In [ ]:
blank_counts = (df == "").sum()

blank_counts[blank_counts > 0].sort_values(ascending=False)

In [ ]:
# ============================================================
# MYSQL CONFIGURATION
# ============================================================

DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "risk_credit_analytics"
DB_USER = "root"
DB_PASSWORD = os.getenv("MYSQL_PASSWORD")
if not DB_PASSWORD:
    raise RuntimeError("Set MYSQL_PASSWORD before running this notebook.")

TABLE_NAME = "credit_risk_raw"

In [ ]:
connection_url = URL.create(
    drivername="mysql+mysqlconnector",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(connection_url)

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    database = conn.execute(text("SELECT DATABASE()")).scalar()

print("Connection test :", result.scalar())
print("Current database:", database)

In [ ]:
create_raw_table_sql = """
CREATE TABLE credit_risk_raw (
    client_ID VARCHAR(50),
    person_age VARCHAR(50),
    person_income VARCHAR(50),
    person_home_ownership VARCHAR(50),
    person_emp_length VARCHAR(50),
    loan_intent VARCHAR(50),
    loan_grade VARCHAR(10),
    loan_amnt VARCHAR(50),
    loan_int_rate VARCHAR(50),
    loan_status VARCHAR(10),
    loan_percent_income VARCHAR(50),
    cb_person_default_on_file VARCHAR(10),
    cb_person_cred_hist_length VARCHAR(50),
    gender VARCHAR(20),
    marital_status VARCHAR(30),
    education_level VARCHAR(50),
    country VARCHAR(50),
    state VARCHAR(100),
    city VARCHAR(100),
    city_latitude VARCHAR(50),
    city_longitude VARCHAR(50),
    employment_type VARCHAR(50),
    loan_term_months VARCHAR(50),
    loan_to_income_ratio VARCHAR(50),
    other_debt VARCHAR(50),
    debt_to_income_ratio VARCHAR(50),
    open_accounts VARCHAR(50),
    credit_utilization_ratio VARCHAR(50),
    past_delinquencies VARCHAR(50)
);
"""

In [ ]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS credit_risk_raw"))
    conn.execute(text(create_raw_table_sql))

print("credit_risk_raw created successfully.")

In [ ]:
query = """
SELECT
    COLUMN_NAME,
    DATA_TYPE,
    CHARACTER_MAXIMUM_LENGTH
FROM information_schema.columns
WHERE table_schema = 'risk_credit_analytics'
  AND table_name = 'credit_risk_raw'
ORDER BY ORDINAL_POSITION;
"""

schema_check = pd.read_sql_query(query, engine)

schema_check

In [ ]:
df.to_sql(
    name=TABLE_NAME,
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

print("Import completed successfully.")

In [ ]:
with engine.connect() as conn:
    mysql_rows = conn.execute(
        text("SELECT COUNT(*) FROM credit_risk_raw")
    ).scalar()

csv_rows = len(df)

print(f"CSV rows   : {csv_rows:,}")
print(f"MySQL rows : {mysql_rows:,}")

In [ ]:
if csv_rows == mysql_rows:
    print("Row count validation: PASSED")
else:
    print("Row count validation: FAILED")

In [ ]:
query = """
SELECT *
FROM credit_risk_raw
LIMIT 10;
"""

df_mysql = pd.read_sql_query(query, engine)

df_mysql

In [ ]:
query = """
SELECT
    SUM(person_emp_length = '') AS blank_emp_length,
    SUM(loan_int_rate = '') AS blank_interest_rate
FROM credit_risk_raw;
"""

pd.read_sql_query(query, engine)